<a href="https://colab.research.google.com/github/JoseAlberto88/Predictive-Maintenance-Dataset/blob/main/Predictive_Maintenance_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DAMO-510-8. Predictive Analytics
## Take-Home Assignment 2: Predictive Maintenance. Machine Failure Prediction


**Course:** Predictive Analytics (DAMO-510-8)

**Term:** Summer 2026

**Instructor:** Prof. Payman Janbakhsh

**Student:** Jose Alberto Martinez Morales

---

## Background

Unexpected machine failures can cause production delays, increased maintenance costs, safety concerns, and operational disruption. Predictive analytics can support maintenance planning by identifying machines that exhibit operating conditions associated with a higher risk of failure.

In this assignment, you will develop and critically evaluate classification models using the AI4I 2020 Predictive Maintenance Dataset.

The purpose of this assignment is not simply to obtain the highest model accuracy. You are expected to demonstrate appropriate analytical judgement regarding data preparation, variable selection, class imbalance, model development, evaluation, interpretation, and operational decision-making.

## Dataset

Use the AI4I 2020 Predictive Maintenance Dataset from the UCI Machine Learning Repository:

https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset

**Target variable:** `Machine failure`

You must use the original dataset obtained from the stated source.


In [1]:
# For this assignment, I will save the Maintenance Dataset in my Google drive and call it using code.
# If you want to run the jupyter notebook in your local machine, please change the directory described in the next code block.

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

DATA_FILE = '/content/drive/MyDrive/Predictive Analytics Course/Maintenance Dataset.csv'
df = pd.read_csv(DATA_FILE)

# To check the 5 first observations of the dataset
print("Initial shape:", df.shape)
df.head()

Initial shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [5]:
df['Machine failure'].value_counts()

,count
Machine failure,
0,9661
1,339


## Assignment Tasks

### Task 1. Problem Definition

Define the predictive problem represented by the dataset.

Your discussion must:

- clearly identify the target variable and classes;
- explain the purpose of predicting machine failure;
- identify potential users of such a prediction system;
- distinguish between the operational consequences of false-positive and false-negative predictions; and
- identify the evaluation criteria you believe should receive the greatest emphasis in this problem.

Support your choices using the context of predictive maintenance rather than generic definitions of classification.

## Task 1. Problem Definition

### **Explanation: Target Variables and Classes**
The AI4I 2020 dataset frames a binary classification problem centred on the variable **Machine failure**, which takes the value 1 when a machine tool has experienced at least one of five recorded failure modes during an operating cycle (tool wear failure, heat dissipation failure, power failure, overstrain failure, or random failure) and 0 when it has not. Of the 10,000 recorded observations, failures are a small minority of cases, which already signals that this is a rare-event detection problem rather than a balanced discrimination task, a point that shapes every later modelling decision, from the choice of resampling strategy to the metrics used to judge success.

### **Explanation: Purpose of Predicting Machine Failure**
The purpose of predicting machine failure is to shift maintenance activity away from the two conventional regimes that dominate manufacturing operations. Purely reactive maintenance repairs a machine only after it has broken, which guarantees unplanned downtime, and time-based preventive maintenance services equipment on a fixed schedule regardless of its actual condition, which wastes labour and parts on machines that did not need attention while still missing failures that occur between scheduled visits. A predictive model that flags an elevated failure risk from live operating conditions, such as temperatures, rotational speed, torque, and tool wear,  allows maintenance to be triggered by the actual state of the machine rather than by the calendar or by failure itself, which is the basic value proposition of condition-based, predictive maintenance.


### **Identification: Potential Users of such a Prediction System**
The natural users of such a system are the people who plan and execute maintenance work rather than the data science team that builds it: maintenance and reliability engineers who would use the model's output to prioritise inspection queues, production planners and plant managers who need advance warning to schedule downtime without disrupting delivery commitments, shop-floor technicians who would receive an operational alert rather than a probability score, and, indirectly, safety personnel, since several of the underlying failure modes (heat dissipation, overstrain) are also safety-relevant conditions. In practice, the model's output would most plausibly be consumed through a maintenance dashboard or a computerised maintenance management system (CMMS) rather than read directly as raw model output.

### **Distintion: False-Positive and False-Negative Predictions **
Because this is a binary decision problem applied to a physical process, the two error types are not interchangeable, and treating them as symmetric would misrepresent the operational reality the model is meant to serve. A **false positive**, it means the model predicts failure when the machine is in fact healthy, triggers an unnecessary inspection or maintenance stoppage. This has a real but bounded cost: lost production time, technician labour, and, if false alarms are frequent, an erosion of operator trust that can lead staff to start ignoring the system altogether (alarm fatigue), which would defeat its purpose over time. A **false negative**, it means the model predicts no failure when the machine is about to fail, is materially more costly in most predictive-maintenance settings: it means the failure is discovered only when it actually occurs, with the same unplanned downtime, cascading equipment damage, and safety exposure that predictive maintenance was introduced to avoid, plus the reputational cost of a system that failed to warn when it was supposed to. Because the dataset also implies a low base rate of failure, the asymmetry compounds: with failures scarce and their cost to miss high, a model that defaults toward the majority "no failure" prediction can look accurate while providing little of the protection the business actually needs.

### **Identification and Evaluation Criteria**
This asymmetry is what should drive the choice of evaluation criteria, and it argues against using overall accuracy as the primary yardstick, since with failures forming a small share of observations a trivial always-predict-no-failure model would already score highly on accuracy while being operationally useless. The metric that should receive the greatest emphasis is **recall on the failure class** (the proportion of actual failures the model successfully flags), because in this problem missing a failure is more expensive than issuing an avoidable inspection. Recall should not be optimised in isolation, however: a model that maximises recall by flagging almost everything as high-risk would generate so many false alarms that maintenance staff could not act on them meaningfully, so **precision** and a metric that balances the two (F1, or an F-beta score that weights recall above precision to reflect the cost asymmetry) should be tracked alongside it, together with a threshold-independent measure such as **ROC-AUC** to assess the model's underlying ability to separate failing from healthy machines before any specific operating threshold is chosen. This combination, recall-weighted performance on the minority class, checked against precision to keep false alarms manageable, and summarised by a threshold-independent ranking metric, reflects how the model would actually be judged and operated in a predictive-maintenance context, rather than how a classifier is conventionally scored in a balanced, generic setting.
